**Défi d’adaptation : un cas nouveau, traité seul**

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping


# Génération des 600 points
X, y = make_moons(n_samples=600, noise=0.35, random_state=42)

Etape 1 : Découpage des données

In [7]:
# On garde 20% pour le test final. stratify=y garde la proportion des classes.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Points pour apprendre (et valider) :", len(X_train))
print("Points pour le test final :", len(X_test))

Points pour apprendre (et valider) : 480
Points pour le test final : 120


Etape 2 : Mise à l'échelle sans fuite

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # Apprend sur train ET transforme
X_test_scaled = scaler.transform(X_test)       # Applique seulement sur test

Etape 3: La référence non neuronale

In [9]:
# 1. Modèle de référence
reference = LogisticRegression()
reference.fit(X_train_scaled, y_train)
y_pred_ref = reference.predict(X_test_scaled)

# 2. Mesures sur le test
accuracy_ref = accuracy_score(y_test, y_pred_ref)
f1_ref = f1_score(y_test, y_pred_ref, average='macro')
matrice_ref = confusion_matrix(y_test, y_pred_ref)

print("=== Méthode de Référence (Régression Logistique) ===")
print("Accuracy :", round(accuracy_ref, 3))
print("F1 Macro :", round(f1_ref, 3))
print("Matrice de confusion :\\n", matrice_ref)

=== Méthode de Référence (Régression Logistique) ===
Accuracy : 0.85
F1 Macro : 0.85
Matrice de confusion :\n [[50 10]
 [ 8 52]]


Etape 4: Entraîner un réseau

In [10]:
# Configuration de l'arrêt précoce
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# création d'un dictionnaire vide pour stocker les prédictions de nos expériences
predictions = {}

print("=== Arrêt précoce défini et dictionnaire prêt ===")

=== Arrêt précoce défini et dictionnaire prêt ===


Etape 5 : L'expérience: comparaison de deux réglages sur 3 graines

In [17]:
# ----- Les réglages : c'est ICI que vous changez les valeurs -----
nb_neurones = 32
graine = 0
taux_dropout = 0.2

# ----- Le réseau (ne rien changer ) -----
keras.utils.set_random_seed(graine)                       # Fixe le hasard pour cette graine
reseau = keras.Sequential()                               # Un réseau vide
reseau.add(layers.Input(shape=(2,)))                      # L'entrée : 2 mesures
reseau.add(layers.Dense(nb_neurones, activation="relu"))  # La couche cachée
reseau.add(layers.Dropout(taux_dropout))                  # Le dropout
reseau.add(layers.Dense(1, activation="sigmoid"))         # Sortie binaire
reseau.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# ----- L'entraînement -----
# On utilise l'arret_precoce défini à l'étape 4
histoire = reseau.fit(X_train_scaled, y_train, validation_split=0.2, epochs=100, batch_size=32, callbacks=[early_stopping], verbose=0)

# ----- Le résultat  -----
probas = reseau.predict(X_test_scaled, verbose=0)
y_pred_res = (probas > 0.5).astype(int).ravel()

accuracy_res = accuracy_score(y_test, y_pred_res)
f1_res = f1_score(y_test, y_pred_res, average='macro')
matrice_res = confusion_matrix(y_test, y_pred_res)

print(f"=== RÉSULTAT (Neurones: {nb_neurones} | Graine: {graine}) ===")
print("Accuracy :", round(accuracy_res, 3))
print("F1 Macro :", round(f1_res, 3))
print("Matrice de confusion :\n", matrice_res)

=== RÉSULTAT (Neurones: 32 | Graine: 0) ===
Accuracy : 0.608
F1 Macro : 0.572
Matrice de confusion :
 [[19 41]
 [ 6 54]]


=== RÉSULTAT (Neurones: 8 | Graine: 42) ===
Accuracy : 0.5
F1 Macro : 0.333
Matrice de confusion :
 [[60  0]
 [60  0]]

 === RÉSULTAT (Neurones: 8 | Graine: 123) ===
Accuracy : 0.792
F1 Macro : 0.791
Matrice de confusion :
 [[45 15]
 [10 50]]

=== RÉSULTAT (Neurones: 8 | Graine: 999) ===
Accuracy : 0.658
F1 Macro : 0.658
Matrice de confusion :
 [[42 18]
 [23 37]]

=== RÉSULTAT (Neurones: 32 | Graine: 42) ===
Accuracy : 0.875
F1 Macro : 0.875
Matrice de confusion :
 [[51  9]
 [ 6 54]]

 === RÉSULTAT (Neurones: 32 | Graine: 123) ===
Accuracy : 0.692
F1 Macro : 0.673
Matrice de confusion :
 [[27 33]
 [ 4 56]]

 === RÉSULTAT (Neurones: 32 | Graine: 999) ===
Accuracy : 0.525
F1 Macro : 0.399
Matrice de confusion :
 [[ 4 56]
 [ 1 59]]




Etape 6 : Mesurer (Accuracy, F1 macro, Matrice)

In [18]:
# Le modèle donne ses probabilités (entre 0 et 1) pour les points du test
probas = reseau.predict(X_test_scaled, verbose=0)

# On transforme ces probabilités en classes fermes (0 ou 1)
# Si c'est supérieur à 0.5, ça devient 1. Sinon ça devient 0.
y_pred_res = (probas > 0.5).astype(int).ravel()

# On sort le bulletin de notes en comparant les prédictions (y_pred_res) avec la réalité (y_test)
accuracy_res = accuracy_score(y_test, y_pred_res)
f1_res = f1_score(y_test, y_pred_res, average='macro')
matrice_res = confusion_matrix(y_test, y_pred_res)

# On affiche les résultats
print("Accuracy :", round(accuracy_res, 3))
print("F1 Macro :", round(f1_res, 3))
print("Matrice de confusion :\n", matrice_res)

Accuracy : 0.608
F1 Macro : 0.572
Matrice de confusion :
 [[19 41]
 [ 6 54]]


Etape 7: Calculer la marge de bruit

In [20]:
# p (précision de la référence) = 0.85
# n (taille du test) = 120 (20% de 600)

# calcul de la marge de bruit selon la formule donnée
bruit = 1.96 * np.sqrt((0.85 * (1 - 0.85)) / 120)

print("Marge de bruit :", round(bruit, 3))

Marge de bruit : 0.064


Conclusion du TP

L'objectif de ce défi était de déterminer si un réseau de neurones surpasse une régression logistique sur un jeu de 600 points non séparables linéairement, et de garantir ce résultat statistiquement.

Le réseau fait-il mieux ?  

De manière générale, non. Le réseau de neurones s'est montré très instable sur ce faible volume de données. Selon les poids de départ, l'Accuracy a chuté jusqu'à 0.500, le modèle se contentant de prédire une seule classe en permanence. En moyenne, nos réseaux affichent une Accuracy d'environ 65 % à 70 %, ce qui est nettement inférieur à la méthode de référence qui atteint 85 %.
Cependant, notre meilleur réglage (32 neurones, graine 42) a réussi à obtenir un score de 87,5 %, battant ainsi la référence avec un gain de +2,5 %.

Peut-on le garantir ?  

Non, nous ne pouvons pas le garantir. Sur un échantillon de test de 120 points et une précision de référence de 85 %, la marge de bruit statistique s'élève à 6,4 %. Le gain apporté par notre meilleur réseau (+2,5 %) est largement inférieur à cette marge de bruit.

*Recommandation* :  

L'amélioration observée avec le réseau n'est pas statistiquement significative et relève de la fluctuation aléatoire. De plus, la forte instabilité de l'entraînement le rend peu fiable ici. Il est donc recommandé au client de conserver la méthode simple (régression logistique), qui est plus stable, ou d'augmenter drastiquement le nombre de données fournies pour justifier l'usage du deep learning.